<a href="https://colab.research.google.com/github/Rahmatul-Rovi/Capstone-Project-Reports/blob/main/Alzheimers_Disease.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

rahmatulrovi386_alzheimers_dataset_path = kagglehub.dataset_download('rahmatulrovi386/alzheimers-dataset')

print('Data source import complete.')


In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve, confusion_matrix,
                             ConfusionMatrixDisplay)

try:
    from xgboost import XGBClassifier
except ImportError:

    raise

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

os.makedirs("results", exist_ok=True)
print("Setup done.")

Setup done.


In [ ]:
CSV_NAME = "/kaggle/input/datasets/rahmatulrovi386/alzheimers-dataset/alzheimers_prediction_dataset.csv"

if not os.path.exists(CSV_NAME):
    try:
        from google.colab import files
        print("Upload the CSV file")
        files.upload()
    except ImportError:
        raise FileNotFoundError(f"{CSV_NAME} Not in the folder. Keep it on the Notebook side.")

df = pd.read_csv(CSV_NAME)
print("Shape:", df.shape)
df.head()

Shape: (74283, 25)


,Country,Age,Gender,Education Level,BMI,Physical Activity Level,Smoking Status,Alcohol Consumption,Diabetes,Hypertension,...,Dietary Habits,Air Pollution Exposure,Employment Status,Marital Status,Genetic Risk Factor (APOE-ε4 allele),Social Engagement Level,Income Level,Stress Levels,Urban vs Rural Living,Alzheimer’s Diagnosis
0,Spain,90,Male,1,33.0,Medium,Never,Occasionally,No,No,...,Healthy,High,Retired,Single,No,Low,Medium,High,Urban,No
1,Argentina,72,Male,7,29.9,Medium,Former,Never,No,No,...,Healthy,Medium,Unemployed,Widowed,No,High,Low,High,Urban,No
2,South Africa,86,Female,19,22.9,High,Current,Occasionally,No,Yes,...,Average,Medium,Employed,Single,No,Low,Medium,High,Rural,No
3,China,53,Male,17,31.2,Low,Never,Regularly,Yes,No,...,Healthy,Medium,Retired,Single,No,High,Medium,Low,Rural,No
4,Sweden,58,Female,3,30.0,High,Former,Never,Yes,No,...,Unhealthy,High,Employed,Married,No,Low,Medium,High,Rural,No


In [ ]:
# Find the Target column
TARGET = [c for c in df.columns if "Diagnosis" in c][0]
print("Target column:", repr(TARGET))

print("\nMissing values :", int(df.isna().sum().sum()))
print("Duplicate rows :", int(df.duplicated().sum()))

print("\nTarget distribution:")
print(df[TARGET].value_counts())
print(df[TARGET].value_counts(normalize=True).round(3))

Target column: 'Alzheimer’s Diagnosis'

Missing values : 0
Duplicate rows : 0

Target distribution:
Alzheimer’s Diagnosis
No     43570
Yes    30713
Name: count, dtype: int64
Alzheimer’s Diagnosis
No     0.587
Yes    0.413
Name: proportion, dtype: float64


In [ ]:
# Column type Changes
y = (df[TARGET] == "Yes").astype(int)          # Yes -> 1, No -> 0
X = df.drop(columns=[TARGET])

numeric_cols     = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()

print("Numeric columns    :", numeric_cols)
print("Categorical columns:", len(categorical_cols))
print(categorical_cols)

# For Report summary table (Chapter 4.2)
summary = pd.DataFrame({
    "Item": ["Total rows", "Total columns (features)", "Numeric features",
             "Categorical features", "Positive (Yes) rate", "Missing values", "Duplicates"],
    "Value": [len(df), X.shape[1], len(numeric_cols), len(categorical_cols),
              f"{y.mean():.3f}", int(df.isna().sum().sum()), int(df.duplicated().sum())]
})
summary.to_csv("results/dataset_summary.csv", index=False)
summary

Numeric columns    : ['Age', 'Education Level', 'BMI', 'Cognitive Test Score']
Categorical columns: 20
['Country', 'Gender', 'Physical Activity Level', 'Smoking Status', 'Alcohol Consumption', 'Diabetes', 'Hypertension', 'Cholesterol Level', 'Family History of Alzheimer’s', 'Depression Level', 'Sleep Quality', 'Dietary Habits', 'Air Pollution Exposure', 'Employment Status', 'Marital Status', 'Genetic Risk Factor (APOE-ε4 allele)', 'Social Engagement Level', 'Income Level', 'Stress Levels', 'Urban vs Rural Living']


,Item,Value
0,Total rows,74283
1,Total columns (features),24
2,Numeric features,4
3,Categorical features,20
4,Positive (Yes) rate,0.413
5,Missing values,0
6,Duplicates,0


In [ ]:
USE_COUNTRY_AS_FEATURE = False

country = df["Country"].copy()

if not USE_COUNTRY_AS_FEATURE:
    X = X.drop(columns=["Country"])
    categorical_cols = [c for c in categorical_cols if c != "Country"]

print("Final feature count:", X.shape[1])

Final feature count: 23
